# 3.4 Practical task

In [1]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy
import re
import pandas as pd
import matplotlib.pyplot as plt

### Load data

In [2]:
bbc_data = pd.read_csv("bbc_news.csv") # the data should be in the same folder as your notebook

In [4]:
bbc_data[15:25]

,Unnamed: 0,index,title,pubDate,guid,link,description
15,15,4054,Top student loan interest rate cut by 5% in En...,"Fri, 10 Jun 2022 23:32:29 GMT",https://www.bbc.co.uk/news/education-61765891,https://www.bbc.co.uk/news/education-61765891?...,"The government says the 7.3% cap offers ""peace..."
16,16,9998,Russians said they’d take my baby: A medic’s s...,"Sat, 05 Nov 2022 00:13:29 GMT",https://www.bbc.co.uk/news/world-europe-63464868,https://www.bbc.co.uk/news/world-europe-634648...,A Ukrainian woman suffered months of poor trea...
17,17,4338,Week in pictures: 11-17 June 2022,"Fri, 17 Jun 2022 23:42:35 GMT",https://www.bbc.co.uk/news/in-pictures-61833427,https://www.bbc.co.uk/news/in-pictures-6183342...,A selection of powerful images from all over t...
18,18,4825,Xi Jinping in Hong Kong: Tight security as cit...,"Fri, 01 Jul 2022 01:54:15 GMT",https://www.bbc.co.uk/news/world-asia-china-61...,https://www.bbc.co.uk/news/world-asia-china-61...,China's leader Xi Jinping is visiting in his f...
19,19,13609,Rescues carried out in silence to listen for s...,"Wed, 08 Feb 2023 21:21:59 GMT",https://www.bbc.co.uk/news/world-europe-64570463,https://www.bbc.co.uk/news/world-europe-645704...,"Days after Turkey's devastating earthquake, th..."
20,20,13042,Nick Pope: Newcastle goalkeeper extends incred...,"Tue, 24 Jan 2023 23:40:46 GMT",https://www.bbc.co.uk/sport/football/64395032,https://www.bbc.co.uk/sport/football/64395032?...,Nick Pope's £10m transfer fee is looking like ...
21,21,3869,How China plans to become the next big space p...,"Sun, 05 Jun 2022 23:01:47 GMT",https://www.bbc.co.uk/news/world-asia-china-61...,https://www.bbc.co.uk/news/world-asia-china-61...,China plans to put astronauts on the Moon and ...
22,22,714,Barcelona thrash leaders Real Madrid in famous...,"Sun, 20 Mar 2022 23:09:29 GMT",https://www.bbc.co.uk/sport/football/60816848,https://www.bbc.co.uk/sport/football/60816848?...,Pierre-Emerick Aubameyang scores twice on his ...
23,23,14615,Kuenssberg: The Budget cannot mask big changes...,"Sat, 11 Mar 2023 16:03:22 GMT",https://www.bbc.co.uk/news/uk-politics-64926786,https://www.bbc.co.uk/news/uk-politics-6492678...,Both main parties are under pressure to make t...
24,24,13584,Nicola Bulley search: What we know about her d...,"Thu, 09 Feb 2023 13:38:27 GMT",https://www.bbc.co.uk/news/uk-england-lancashi...,https://www.bbc.co.uk/news/uk-england-lancashi...,The missing mum vanished on a riverside dog wa...


In [8]:
bbc_data[15:25].info()
bbc_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 15 to 24
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   10 non-null     int64 
 1   index        10 non-null     int64 
 2   title        10 non-null     object
 3   pubDate      10 non-null     object
 4   guid         10 non-null     object
 5   link         10 non-null     object
 6   description  10 non-null     object
dtypes: int64(2), object(5)
memory usage: 692.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1000 non-null   int64 
 1   index        1000 non-null   int64 
 2   title        1000 non-null   object
 3   pubDate      1000 non-null   object
 4   guid         1000 non-null   object
 5   link         1000 non-null   object
 6   description  1000 non-null   object

In [9]:
titles = pd.DataFrame(bbc_data['title'])
titles.head()

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...
2,Rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...


### Clean data

In [10]:
# lowercase
titles['lowercase'] = titles['title'].str.lower()
titles.head()

,title,lowercase
0,Can I refuse to work?,can i refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...


In [11]:
# stop word removal
en_stopwords = stopwords.words('english')
titles['no_stopwords'] = titles['lowercase'].apply(lambda x: ' '.join([word for word in x.split() if word not in (en_stopwords)]))
titles.head()

,title,lowercase,no_stopwords
0,Can I refuse to work?,can i refuse to work?,refuse work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds


In [12]:
# punctation removal
titles['no_stopwords_no_punct'] = titles.apply(lambda x: re.sub(r"([^\w\s])", "", x['no_stopwords']), axis=1)
titles.head()

,title,lowercase,no_stopwords,no_stopwords_no_punct
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds


In [13]:
# tokenize
titles['tokens_raw'] = titles.apply(lambda x: word_tokenize(x['title']), axis=1)
titles['tokens_clean'] = titles.apply(lambda x: word_tokenize(x['no_stopwords_no_punct']), axis=1)
titles.head()

,title,lowercase,no_stopwords,no_stopwords_no_punct,tokens_raw,tokens_clean
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[Can, I, refuse, to, work, ?]","[refuse, work]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"['Liz, Truss, the, Brief, ?, ', World, reacts,...","[liz, truss, brief, world, reacts, uk, politic..."
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[Rationing, energy, is, nothing, new, for, off...","[rationing, energy, nothing, new, offgrid, com..."
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs,"[The, hunt, for, superyachts, of, sanctioned, ...","[hunt, superyachts, sanctioned, russian, oliga..."
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds,"[Platinum, Jubilee, :, 70, years, of, the, Que...","[platinum, jubilee, 70, years, queen, 70, seco..."


In [14]:
# lemmatizing 
lemmatizer = WordNetLemmatizer()
titles["tokens_clean_lemmatized"] = titles["tokens_clean"].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])
titles.head()

,title,lowercase,no_stopwords,no_stopwords_no_punct,tokens_raw,tokens_clean,tokens_clean_lemmatized
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[Can, I, refuse, to, work, ?]","[refuse, work]","[refuse, work]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"['Liz, Truss, the, Brief, ?, ', World, reacts,...","[liz, truss, brief, world, reacts, uk, politic...","[liz, truss, brief, world, reacts, uk, politic..."
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[Rationing, energy, is, nothing, new, for, off...","[rationing, energy, nothing, new, offgrid, com...","[rationing, energy, nothing, new, offgrid, com..."
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs,"[The, hunt, for, superyachts, of, sanctioned, ...","[hunt, superyachts, sanctioned, russian, oliga...","[hunt, superyachts, sanctioned, russian, oliga..."
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds,"[Platinum, Jubilee, :, 70, years, of, the, Que...","[platinum, jubilee, 70, years, queen, 70, seco...","[platinum, jubilee, 70, year, queen, 70, second]"


In [16]:
# create lists for just our tokens
tokens_raw_list = sum(titles['tokens_raw'], []) #unpack our lists into a single list
tokens_clean_list = sum(titles['tokens_clean_lemmatized'], [])
print(tokens_raw_list[:10])
print(tokens_clean_list[:10])

['Can', 'I', 'refuse', 'to', 'work', '?', "'Liz", 'Truss', 'the', 'Brief']
['refuse', 'work', 'liz', 'truss', 'brief', 'world', 'reacts', 'uk', 'political', 'turmoil']


### POS Tagging

In [17]:
nlp = spacy.load('en_core_web_sm')

In [28]:
# create a spacy doc from our raw text - better for pos tagging
spacy_doc = nlp(' '.join(tokens_raw_list))
print(spacy_doc[:10])
clean_spacy_doc = nlp(' '.join(tokens_clean_list))
print(clean_spacy_doc[:10])


Can I refuse to work ? 'Liz Truss the
refuse work liz truss brief world reacts uk political turmoil


In [33]:
# extract the tokens and pos tags into a dataframe
pos_df = pd.DataFrame(columns=['token', 'pos_tag'])

for token in spacy_doc:
    pos_df = pd.concat([pos_df,
                       pd.DataFrame.from_records([{'token': token.text,'pos_tag': token.pos_}])], ignore_index=True)
    
print("raw", pos_df.head())

clean_pos_df = pd.DataFrame(columns=['token', 'pos_tag'])
for token in clean_spacy_doc:
    clean_pos_df = pd.concat([clean_pos_df,
                              pd.DataFrame.from_records([{'token': token.text,'pos_tag': token.pos_}])], ignore_index=True)

print("clean", clean_pos_df.head())

raw     token pos_tag
0     Can     AUX
1       I    PRON
2  refuse    VERB
3      to    PART
4    work    VERB
clean     token pos_tag
0  refuse     AUX
1    work    NOUN
2     liz   PROPN
3   truss     ADJ
4   brief     ADJ


In [37]:
# token frequency count
pos_df_counts = pos_df.groupby(['token','pos_tag']).size().reset_index(name='counts').sort_values(by='counts', ascending=False)
print(pos_df_counts.head(10))

clean_pos_df_counts = clean_pos_df.groupby(['token','pos_tag']).size().reset_index(name='counts').sort_values(by='counts', ascending=False)
print(clean_pos_df_counts.head(10))

     token pos_tag  counts
95       :   PUNCT     543
8        '   PUNCT     300
2897    in     ADP     187
4082    to    PART     175
3268    of     ADP     172
22       -   PUNCT     166
4043   the     DET     163
1856   and   CCONJ     147
15      's    PART     143
97       ?   PUNCT     130
        token pos_tag  counts
30       2022     NUM      47
1162  england   PROPN      45
870       cup   PROPN      39
3056      say    VERB      37
3707       uk   PROPN      37
3840      war    NOUN      34
2386      new     ADJ      31
3948    world    NOUN      30
3949    world   PROPN      26
3710  ukraine   PROPN      23


In [39]:
# most common nouns
nouns = pos_df_counts[pos_df_counts.pos_tag == "NOUN"][0:10]
print(nouns)

clean_nouns = clean_pos_df_counts[clean_pos_df_counts.pos_tag == "NOUN"][0:10]
print(clean_nouns)

       token pos_tag  counts
4267     war    NOUN      35
3552  record    NOUN      15
3416  police    NOUN      14
4356    year    NOUN      14
4316     win    NOUN      14
3061  living    NOUN      13
4009     tax    NOUN      13
2326     day    NOUN      12
3368  people    NOUN      12
2031    boss    NOUN      11
         token pos_tag  counts
3840       war    NOUN      34
3948     world    NOUN      30
2136       man    NOUN      22
907        day    NOUN      21
3973      year    NOUN      20
1158    energy    NOUN      17
2847    record    NOUN      17
3935     woman    NOUN      16
1130  election    NOUN      16
3870      week    NOUN      16


In [25]:
# most common verbs
verbs = pos_df_counts[pos_df_counts.pos_tag == "VERB"][0:10]
verbs

,token,pos_tag,counts
3687,says,VERB,30
9,',VERB,14
2670,found,VERB,13
4317,win,VERB,12
4324,wins,VERB,10
2713,get,VERB,9
2388,dies,VERB,9
3990,take,VERB,8
2982,killed,VERB,8
3686,say,VERB,8


In [26]:
# most common adjectives
adj = pos_df_counts[pos_df_counts.pos_tag == "ADJ"][0:10]
adj

,token,pos_tag,counts
3244,new,ADJ,28
1400,Russian,ADJ,21
2606,final,ADJ,16
19,-,ADJ,14
2625,first,ADJ,12
3199,more,ADJ,10
1994,big,ADJ,9
2835,high,ADJ,9
3000,last,ADJ,8
3304,other,ADJ,8


### NER

In [41]:
# extract the tokens and entity tags into a dataframe
ner_df = pd.DataFrame(columns=['token', 'ner_tag'])

print(spacy_doc.ents[:10])

for token in spacy_doc.ents:
    if pd.isna(token.label_) is False:
        ner_df = pd.concat([ner_df, pd.DataFrame.from_records(
            [{'token': token.text, 'ner_tag': token.label_}])], ignore_index=True)

(Liz Truss, UK, Rationing, superyachts, Russian, 70 years, 70 seconds, Red Bull, Formula 1 's, World Triathlon Championship Series)


In [42]:
ner_df.head()

,token,ner_tag
0,Liz Truss,PERSON
1,UK,GPE
2,Rationing,PRODUCT
3,superyachts,CARDINAL
4,Russian,NORP


In [43]:
# token frequency count
ner_df_counts = ner_df.groupby(['token','ner_tag']).size().reset_index(name='counts').sort_values(by='counts', ascending=False)
ner_df_counts.head(10)

,token,ner_tag,counts
965,Ukraine,GPE,47
955,UK,GPE,36
329,England,GPE,32
819,Russian,NORP,20
957,US,GPE,19
1031,World Cup 2022,EVENT,18
1058,first,ORDINAL,13
918,The Papers,WORK_OF_ART,13
378,France,GPE,12
226,China,GPE,11


In [44]:
# most common people
people = ner_df_counts[ner_df_counts.ner_tag == "PERSON"][0:10]
people

,token,ner_tag,counts
257,Covid,PERSON,9
760,Queen,PERSON,8
757,Putin,PERSON,8
169,Boris Johnson,PERSON,6
563,Liz Truss,PERSON,6
788,Rishi Sunak,PERSON,5
581,Macron,PERSON,4
762,Quiz,PERSON,4
515,Jurgen Klopp,PERSON,4
325,Emma Raducanu,PERSON,4


In [45]:
# most common places
places = ner_df_counts[ner_df_counts.ner_tag == "GPE"][0:10]
places

,token,ner_tag,counts
965,Ukraine,GPE,47
955,UK,GPE,36
329,England,GPE,32
957,US,GPE,19
378,France,GPE,12
226,China,GPE,11
817,Russia,GPE,9
454,India,GPE,8
132,Australia,GPE,7
566,London,GPE,7
